In [ ]:
# Esta función genera un ciclo con los puntos 1) y 2).
# Ejecuta GWF-MT3D-NVG.ipynb haciendo 1): 
# 1) Lee conc_s.txt para modificar transport.btn y ejecutar GWF-MT3D-NVG.ipynb para generar conc_i.m3d.
# 2) Ejecuta RT.exe que lee conc_i.m3d, lo modifica y genera conc_s.txt
# 3) Se repite el ciclo [1),2)]
#******************************************************************************************************
# 1. Importación de librerías
import numpy as np
import subprocess
import time
import ipywidgets as widgets
from IPython.display import display

# 2. Rutas de archivos
#entrada_txt = "conc_s.txt"   # u_new.dat
#salida_m3d = "conc_i.m3d"    # u_tilde
entrada_txt = "u_new.out"     # u_new.out
salida_m3d = "u_tilde.dat"    # u_tilde.dat
main_interfaz = "main_interfaz_equi.exe" 
#ejecutable = "main_interfaz.exe" 

# 3. Slider para controlar nv
nv_slider = widgets.IntSlider(value=1, min=1, max=40, step=1, description="nv")
run_button = widgets.Button(description="Iniciar ciclo ")
output_area = widgets.Output()
contador_ejecuciones = 0

# 4. Función que ejecuta el ciclo
def ciclo_nv(b):
    global contador_ejecuciones
    output_area.clear_output()
    
    with output_area:
        contador_ejecuciones += 1
        print(f"🟢  Ejecución #{contador_ejecuciones}")        
        nv = nv_slider.value

        for i in range(nv):
            print(f"\n--- Iteración {i+1} de {nv} ---")
            
            #XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
            # EJECUTAR: (GWF,MT3D) ó GWF-MT3D-NVG
            # Condición según contador de ejecuciones
            if contador_ejecuciones == 1 and i == 0:
                print("▶️✅ Primera ejecución: corriendo GWF.ipynb y MT3D.ipynb...")
                %run GWF.ipynb  # Ejecutar GWF
                print(" 🟢🟢 Termina GWF, Inicia MT3D")
                %run MT3D.ipynb # Ejecutar MT3D
            else:
                print("▶️✅ Solo corriendo MT3D.ipynb...")
                %run MT3D.ipynb
                #%run GWF-MT3D-NVG.ipynb
                
##########################################################################################
# sustituir_primera_linea.py
# Sustituye el primer renglón de u_tilde.dat por el contenido de conc_i.m3d

def sustituir_primera_linea(archivo_u_tilde="u_tilde.dat", archivo_conc="conc_i.m3d"):
    try:
        # Leer el contenido de conc_i.m3d
        with open(archivo_conc, 'r') as f_conc:
            contenido_conc = f_conc.read().strip()  # puede tener varios valores o líneas
        
        # Leer todas las líneas de u_tilde.dat
        with open(archivo_u_tilde, 'r') as f_ut:
            lineas = f_ut.readlines()
        
        if len(lineas) < 2:
            print(f"⚠️ El archivo {archivo_u_tilde} no tiene al menos dos líneas.")
            return
        
        # Reemplazar la primera línea con el contenido de conc_i.m3d
        lineas[0] = contenido_conc + "\n"
        
        # Escribir nuevamente el archivo
        with open(archivo_u_tilde, 'w') as f_ut:
            f_ut.writelines(lineas)
        
        print(f"✅ Sustitución completada en {archivo_u_tilde}")
        print(f"👉 Primer renglón reemplazado por el contenido de {archivo_conc}")
    
    except Exception as e:
        print(f"❌ Error: {e}")


# Ejecutar directamente (si se corre como script)
if __name__ == "__main__":
    sustituir_primera_linea()


#########################################################################################
# La primera ejecución de MT3D se hace con u_new1.out que son las concentraciones en t=0
# Se ejecuta MT3D-USGS, para crear conc_i.m3d. Sustituir en u_tilde.dat la concentración 
# que llega de MT3D (conc_i.m3d) dejando la segunda parte de u_tilde.dat sin cambio
                
            #XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
                # Ejecutar (main_interfaz_equi.exe)
            try:
                print(f"✅ Ejecutando {main_interfaz} ...\n")
                proceso = subprocess.Popen([main_interfaz], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

                for linea in proceso.stdout:
                    print(linea, end="")  # imprime la salida línea por línea en tiempo real

                proceso.wait()
                if proceso.returncode != 0:
                    print(f"❌ El main_interfaz terminó con error. Código: {proceso.returncode}")
                    return  # o break, si estás dentro de un ciclo
            except Exception as e:
                print(f"❌ Error al ejecutar {main_interfaz}: {e}")
                return  # o break

# Despues de la ejecución de main_interfaz_equi.exe, se obtiene u_new.out 
# De u_new.out, con el primer renglón se obtiene u_new1.out para que las lea el 
# paquete BTN de MT3D y sea posible ejecutar nuevamente el MT3D.
                
            print("✅ Construimos BTN, ejecutamos MT3D-USGS...\n")
            print("⏸️ Pausa de 1 segundo...\n")
            time.sleep(1)

# 5. Conectar botón
run_button.on_click(ciclo_nv)

# 6. Mostrar interfaz
display(nv_slider, run_button, output_area)